# Блок 4 — Прикладные инструменты (DH)

Анализ эмоций в текстах: дуги нарратива, временные ряды, хитмапы, облака слов.

> **Требование:** сначала запусти блоки 01 → 02 → 03.

In [ ]:
import sys, os

PROJECT_ROOT = '/kaggle/input/sentiment-analysis' if os.path.exists('/kaggle') else os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

WORKING_DIR = '/kaggle/working' if os.path.exists('/kaggle') else '../results'

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

from src.data_loader import EKMAN_LABEL_NAMES, EKMAN_ID2LABEL

EMOTION_COLORS = {
    'anger':    '#e74c3c',
    'disgust':  '#8e44ad',
    'fear':     '#2c3e50',
    'joy':      '#f39c12',
    'sadness':  '#3498db',
    'surprise': '#1abc9c',
    'neutral':  '#95a5a6',
}

# Загрузка label_names
lnames_path = f'{WORKING_DIR}/label_names.json'
if os.path.exists(lnames_path):
    with open(lnames_path) as f:
        label_names = json.load(f)
else:
    label_names = EKMAN_LABEL_NAMES
print(f'Классы: {label_names}')

In [ ]:
# Загрузка модели для инференса
# Укажи пути к обученным моделям или HF Hub ID
from src.inference import EmotionClassifier

MODEL_DIRS = [
    f'{WORKING_DIR}/models/rubert',
    f'{WORKING_DIR}/models/xlmroberta',
    f'{WORKING_DIR}/models/rubert_tiny',
]

# Фильтруем только существующие директории
available = [d for d in MODEL_DIRS if os.path.exists(d)]
if not available:
    print('Модели не найдены — используйте ноутбуки 02-04 для обучения.')
    print('Для демонстрации будет использован мок-классификатор.')
    clf = None
else:
    clf = EmotionClassifier(available, clean=True)
    print(f'Загружено моделей: {len(available)}')


def predict_emotions(texts, batch_size=32):
    """Возвращает DataFrame с вероятностями эмоций для каждого текста."""
    if clf is None:
        # Случайный мок для демонстрации без обученных моделей
        rng = np.random.default_rng(42)
        probs = rng.dirichlet(np.ones(len(label_names)), size=len(texts))
        return pd.DataFrame(probs, columns=label_names)
    probs = clf.predict_proba(texts, batch_size=batch_size)
    return pd.DataFrame(probs, columns=label_names)

## 1. Эмоциональные дуги нарратива

Разбиваем текст на предложения (через `razdel`) и строим эмоциональную кривую по ходу повествования.

In [ ]:
try:
    from razdel import sentenize
    HAS_RAZDEL = True
except ImportError:
    print('razdel не установлен: pip install razdel')
    HAS_RAZDEL = False


def split_sentences(text: str):
    if HAS_RAZDEL:
        return [s.text for s in sentenize(text) if len(s.text.strip()) > 10]
    # Простой fallback без razdel
    import re
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.strip()) > 10]


def emotion_arc(text: str, window: int = 3, title: str = '') -> pd.DataFrame:
    """
    Строит эмоциональную дугу текста.
    window: скользящее среднее для сглаживания.
    """
    sentences = split_sentences(text)
    if len(sentences) < 2:
        print('Текст слишком короткий — нужно минимум 2 предложения.')
        return pd.DataFrame()

    df = predict_emotions(sentences)
    df['sentence_idx'] = range(len(df))
    df['dominant'] = df[label_names].idxmax(axis=1)

    # Сглаживание
    smooth = df[label_names].rolling(window, center=True, min_periods=1).mean()

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})

    ax = axes[0]
    for emotion in label_names:
        ax.plot(smooth.index, smooth[emotion], label=emotion,
                color=EMOTION_COLORS[emotion], linewidth=2, alpha=0.85)
    ax.set_ylabel('Вероятность')
    ax.set_title(f'Эмоциональная дуга{" — " + title if title else ""}')
    ax.legend(loc='upper right', ncol=4, fontsize=9)
    ax.set_xlim(0, len(df) - 1)
    ax.grid(alpha=0.3)

    # Нижняя полоска: доминирующая эмоция
    ax2 = axes[1]
    for i, (_, row) in enumerate(df.iterrows()):
        ax2.barh(0, 1, left=i, color=EMOTION_COLORS[row['dominant']], height=0.6)
    ax2.set_xlim(0, len(df))
    ax2.set_yticks([])
    ax2.set_xlabel('Предложение №')
    ax2.set_title('Доминирующая эмоция (по предложениям)')

    # Легенда для полоски
    from matplotlib.patches import Patch
    unique_ems = df['dominant'].unique()
    legend_patches = [Patch(color=EMOTION_COLORS[e], label=e) for e in unique_ems]
    ax2.legend(handles=legend_patches, loc='upper right', ncol=len(unique_ems), fontsize=8)

    plt.tight_layout()
    plt.savefig(f'{WORKING_DIR}/emotion_arc.png', dpi=150, bbox_inches='tight')
    plt.show()
    return df


# ── Пример использования ──────────────────────────────────────────
sample_text = """
Утро началось прекрасно. Солнце светило, птицы пели, и казалось, что весь мир улыбается.
Но потом пришло то самое письмо. Я прочитал его трижды, не веря своим глазам.
Сердце сжалось от страха. Что теперь будет? Как объяснить это коллегам?
Я был в полном отчаянии. Неужели всё, что мы строили годами, рухнет за один день?
Но потом я вспомнил слова своего наставника: «В любом кризисе есть возможность».
Я выпрямился, сделал глубокий вдох и начал думать, что можно сделать прямо сейчас.
К вечеру план был готов. Команда поддержала меня. Мы справимся.
"""

arc_df = emotion_arc(sample_text, window=2, title='Пример нарратива')

## 2. Эмоциональный временной ряд

Для датированного корпуса (новости, посты в соцсетях) строим динамику эмоций по времени.

In [ ]:
def emotion_timeline(
    df: pd.DataFrame,
    text_col: str = 'text',
    date_col: str = 'date',
    freq: str = 'W',        # 'D'=daily, 'W'=weekly, 'M'=monthly
    title: str = ''
) -> pd.DataFrame:
    """
    Строит временной ряд эмоций по датированному корпусу.
    df должен содержать колонки text_col и date_col.
    """
    texts = df[text_col].tolist()
    dates = pd.to_datetime(df[date_col])

    probs_df = predict_emotions(texts)
    probs_df['date'] = dates.values
    probs_df = probs_df.set_index('date')

    # Агрегация по периоду
    ts = probs_df[label_names].resample(freq).mean()

    fig, ax = plt.subplots(figsize=(14, 5))
    for emotion in label_names:
        ax.plot(ts.index, ts[emotion], label=emotion,
                color=EMOTION_COLORS[emotion], linewidth=2)

    ax.set_ylabel('Средняя вероятность')
    ax.set_title(f'Динамика эмоций{" — " + title if title else ""} (агрегация: {freq})')
    ax.legend(ncol=4, loc='upper right', fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{WORKING_DIR}/emotion_timeline.png', dpi=150, bbox_inches='tight')
    plt.show()
    return ts


# ── Пример с синтетическим корпусом ──────────────────────────────
import random
random.seed(42)

SAMPLE_CORPUS = {
    'joy':     ['Отличный день! Мы победили!', 'Замечательные новости от команды.', 'Радость переполняет меня сегодня.'],
    'sadness': ['Потеря невосполнима.', 'Грустно наблюдать за происходящим.', 'Сердце разрывается.'],
    'fear':    ['Страшно думать о последствиях.', 'Неопределённость давит.', 'Боюсь, что всё пойдёт не так.'],
    'anger':   ['Это возмутительно!', 'Терпеть такое невозможно.', 'Они перешли все границы.'],
    'neutral': ['Совещание перенесли на пятницу.', 'Документы подписаны.', 'Отчёт готов к отправке.'],
}

n = 80
all_texts = [t for texts in SAMPLE_CORPUS.values() for t in texts]
corpus_df = pd.DataFrame({
    'text': [random.choice(all_texts) for _ in range(n)],
    'date': pd.date_range('2023-01-01', periods=n, freq='3D'),
    'author': [random.choice(['Автор А', 'Автор Б', 'Автор В']) for _ in range(n)],
})

timeline = emotion_timeline(corpus_df, freq='W', title='Синтетический корпус')

## 3. Тепловая карта эмоций по авторам / документам

In [ ]:
def emotion_heatmap(
    df: pd.DataFrame,
    text_col: str = 'text',
    group_col: str = 'author',
    title: str = ''
) -> pd.DataFrame:
    """
    Тепловая карта: строки = группы (авторы/документы), столбцы = эмоции.
    Значение = средняя вероятность эмоции в группе.
    """
    texts = df[text_col].tolist()
    probs_df = predict_emotions(texts)
    probs_df[group_col] = df[group_col].values

    grouped = probs_df.groupby(group_col)[label_names].mean()

    fig, ax = plt.subplots(figsize=(10, max(4, len(grouped) * 0.7)))
    sns.heatmap(
        grouped, annot=True, fmt='.2f', cmap='YlOrRd',
        ax=ax, linewidths=0.5, vmin=0, vmax=1,
        cbar_kws={'shrink': 0.7},
    )
    ax.set_title(f'Эмоциональный профиль по {group_col}{" — " + title if title else ""}')
    ax.set_xlabel('Эмоция')
    ax.set_ylabel(group_col.capitalize())
    plt.tight_layout()
    plt.savefig(f'{WORKING_DIR}/emotion_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    return grouped


heatmap_df = emotion_heatmap(corpus_df, group_col='author', title='Авторский профиль')
print(heatmap_df.round(3))

## 4. Облака слов по эмоциям

In [ ]:
try:
    from wordcloud import WordCloud
    HAS_WORDCLOUD = True
except ImportError:
    print('wordcloud не установлен: pip install wordcloud')
    HAS_WORDCLOUD = False


def emotion_wordclouds(
    df: pd.DataFrame,
    text_col: str = 'text',
    top_k_emotions: int = 4,
    min_texts: int = 3,
):
    """
    Для каждой эмоции строит облако из слов текстов с этой доминирующей эмоцией.
    """
    if not HAS_WORDCLOUD:
        return

    texts = df[text_col].tolist()
    probs_df = predict_emotions(texts)
    probs_df['text'] = texts
    probs_df['dominant'] = probs_df[label_names].idxmax(axis=1)

    emotions_present = [e for e in label_names
                        if (probs_df['dominant'] == e).sum() >= min_texts][:top_k_emotions]

    ncols = min(4, len(emotions_present))
    nrows = (len(emotions_present) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4))
    axes = np.array(axes).flatten()

    for ax, emotion in zip(axes, emotions_present):
        subset = probs_df[probs_df['dominant'] == emotion]['text']
        corpus = ' '.join(subset.tolist())
        wc = WordCloud(
            width=400, height=300, background_color='white',
            color_func=lambda *args, **kwargs: EMOTION_COLORS[emotion],
            max_words=50, collocations=False,
        ).generate(corpus)
        ax.imshow(wc, interpolation='bilinear')
        ax.set_title(f'{emotion} (n={len(subset)})', fontsize=12)
        ax.axis('off')

    for ax in axes[len(emotions_present):]:
        ax.axis('off')

    plt.suptitle('Облака слов по эмоциям', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{WORKING_DIR}/wordclouds.png', dpi=150, bbox_inches='tight')
    plt.show()


emotion_wordclouds(corpus_df)

## 5. Атрибуция токенов (объяснимость)

Показывает, какие слова больше всего повлияли на предсказанную эмоцию. Использует `transformers-interpret` (Integrated Gradients).

In [ ]:
def explain_prediction(text: str, model_dir: str | None = None):
    """
    Визуализирует вклад токенов в предсказание через Integrated Gradients.
    Требует transformers-interpret: pip install transformers-interpret
    """
    try:
        from transformers_interpret import SequenceClassificationExplainer
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
    except ImportError:
        print('transformers-interpret не установлен: pip install transformers-interpret')
        return

    if model_dir is None:
        model_dir = available[0] if available else None
    if not model_dir or not os.path.exists(model_dir):
        print('Укажи путь к обученной модели в model_dir.')
        return

    print(f'Загружаю модель из {model_dir}...')
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)

    explainer = SequenceClassificationExplainer(model, tokenizer)
    word_attributions = explainer(text)

    predicted_class = label_names[explainer.predicted_class_index]
    print(f'\nПредсказанная эмоция: {predicted_class}')
    print(f'Атрибуции токенов:\n')
    explainer.visualize()
    return word_attributions


# ── Пример ──────────────────────────────────────────────────────
if available:
    attrs = explain_prediction(
        'Мне очень страшно и тревожно от этих новостей.',
        model_dir=available[0],
    )
else:
    print('Нет обученных моделей для объяснимости. Запустите ноутбуки 02-04.')

## 6. Сравнение эмоциональных профилей авторов / жанров

Radar chart (паукообразная диаграмма) для сравнения эмоциональных профилей.

In [ ]:
def radar_chart(
    profiles: dict[str, dict[str, float]],
    title: str = 'Сравнение эмоциональных профилей'
):
    """
    profiles: {'Автор А': {'joy': 0.3, 'sadness': 0.1, ...}, ...}
    """
    emotions = label_names
    N = len(emotions)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]  # замкнуть

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={'polar': True})
    colors = list(mcolors.TABLEAU_COLORS.values())

    for (name, profile), color in zip(profiles.items(), colors):
        values = [profile.get(e, 0) for e in emotions]
        values += values[:1]
        ax.plot(angles, values, 'o-', linewidth=2, label=name, color=color)
        ax.fill(angles, values, alpha=0.1, color=color)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(emotions, fontsize=11)
    ax.set_title(title, pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    ax.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.savefig(f'{WORKING_DIR}/radar_profiles.png', dpi=150, bbox_inches='tight')
    plt.show()


# Вычисляем профили из корпуса
texts = corpus_df['text'].tolist()
probs_all = predict_emotions(texts)
probs_all['author'] = corpus_df['author'].values

profiles = {
    author: probs_all[probs_all['author'] == author][label_names].mean().to_dict()
    for author in corpus_df['author'].unique()
}

radar_chart(profiles, title='Эмоциональные профили авторов')

## 7. Загрузка и анализ собственного корпуса

Загрузи свой CSV-файл с текстами и применяй все функции выше.

In [ ]:
# Укажи путь к своему корпусу
CORPUS_PATH = None  # Например: 'my_corpus.csv'
TEXT_COL    = 'text'
DATE_COL    = 'date'    # или None если нет дат
GROUP_COL   = 'author'  # или 'genre', 'source' и т.д.

if CORPUS_PATH and os.path.exists(CORPUS_PATH):
    my_corpus = pd.read_csv(CORPUS_PATH)
    print(f'Загружен корпус: {len(my_corpus):,} текстов')
    print(my_corpus.head(3))

    # Полный анализ одной командой:
    probs_corpus = predict_emotions(my_corpus[TEXT_COL].tolist())
    probs_corpus['dominant'] = probs_corpus[label_names].idxmax(axis=1)

    print('\nРаспределение доминирующих эмоций:')
    print(probs_corpus['dominant'].value_counts())

    if DATE_COL and DATE_COL in my_corpus.columns:
        my_corpus_dated = my_corpus.copy()
        emotion_timeline(my_corpus_dated, text_col=TEXT_COL, date_col=DATE_COL)

    if GROUP_COL and GROUP_COL in my_corpus.columns:
        emotion_heatmap(my_corpus, text_col=TEXT_COL, group_col=GROUP_COL)
else:
    print('Установи CORPUS_PATH для анализа собственного корпуса.')
    print('Поддерживаемые форматы: CSV с колонкой text (и опционально date, author).')